In [ ]:
!pip install git+https://github.com/KellerJordan/Muon

  Cloning https://github.com/KellerJordan/Muon to /tmp/pip-req-build-0klotw3g
  Running command git clone --filter=blob:none --quiet https://github.com/KellerJordan/Muon /tmp/pip-req-build-0klotw3g
  Resolved https://github.com/KellerJordan/Muon to commit 6399c658d3c4a3356ba823fa6664b10e23871068
  Preparing metadata (setup.py) ... done
  Created wheel for muon-optimizer: filename=muon_optimizer-0.1.0-py3-none-any.whl size=7141 sha256=1b755df82a82a9c045988d1caf15b2c5b96d26bcc342cda223471f8d06a5034b
  Stored in directory: /tmp/pip-ephem-wheel-cache-lag6y6nu/wheels/6e/33/94/64d18603ba0f39064aab523d6edf493c388cfb7419bb5c9043
Successfully built muon-optimizer


In [ ]:
import os

CACHE_DIR_BASE = "/content/my_hf_cache"

os.environ["HF_HOME"] = CACHE_DIR_BASE  # General Hugging Face home

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import torchvision
import torchvision.transforms as transforms
import time
import numpy as np

from datasets import load_dataset
from torch.utils.data import Dataset

from muon import SingleDeviceMuonWithAuxAdam

# ==========================================
# 1. Configuration & Physics
# ==========================================
# OPTIONS: 'baseline', 'weak' (Covariance), 'strong' (LeJEPA/Epps-Pulley)
REG_MODE = 'strong'
SIGR_ALPHA = 0.001
SKETCH_DIM = 64

# ImageNet Standard Configs
BATCH_SIZE = 256
LEARNING_RATE = 4e-3
EPOCHS = 200
WEIGHT_DECAY = 0.05
NUM_CLASSES = 1000
IMG_SIZE = 224
PATCH_SIZE = 16

DROPOUT = 0.0
MIXUP_ALPHA = 0.8
CUTMIX_ALPHA = 1.0

# Scheduling Configs
LOG_INTERVAL = 50
EVAL_INTERVAL = 5000

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

import wandb
wandb.login() # You'll need an API key

wandb.init(
    project="imagenet",
    name=f"{REG_MODE}_SIGReg",
    config={
        "batch_size": BATCH_SIZE,
        "reg_mode": REG_MODE,
        "sigreg_alpha": SIGR_ALPHA,
        "sketch_dim": SKETCH_DIM,
        "optimizer": "Muon"
    }
)

if torch.backends.mps.is_available():
    DEVICE = 'mps'

print(f"Training on: {DEVICE} | Mode: {REG_MODE} | Alpha: {SIGR_ALPHA}")

def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

# ------------------------------------------
# Physics Engine: The Regularizers
# ------------------------------------------

def sigreg_weak_loss(x, sketch_dim=64):
    """
    Forces Covariance(x) ~ Identity.
    Matches the 2nd Moment (Spherical Cloud).
    """
    N, C = x.size()

    # 1. Sketching
    S = torch.randn(sketch_dim, C, device=x.device) / (C ** 0.5)
    x = x @ S.T  # [N, sketch_dim]

    # 2. Centering & Covariance
    x = x - x.mean(dim=0, keepdim=True)
    cov = (x.T @ x) / (N - 1 + 1e-6)

    # 3. Target Identity
    target = torch.eye(sketch_dim, device=x.device)

    # 4. Off-diagonal suppression + Diagonal maintenance
    return torch.norm(cov - target, p='fro')

def sigreg_strong_loss(x, sketch_dim=64):
    """
    Forces ECF(x) ~ ECF(Gaussian).
    Matches ALL Moments (Maximum Entropy Cloud).
    Exact implementation of LeJEPA Algorithm 1.
    """
    N, C = x.size()

    # 1. Projection
    A = torch.randn(C, sketch_dim, device=x.device)
    A = A / (A.norm(p=2, dim=0, keepdim=True) + 1e-6)

    # 2. Integration Points
    t = torch.linspace(-5, 5, 17, device=x.device)
    exp_f = torch.exp(-0.5 * t**2)

    # 3. Empirical CF (Rewritten to avoid complex numbers for torch.compile support)
    proj = x @ A
    args = proj.unsqueeze(2) * t.view(1, 1, -1)

    # exp(ix) = cos(x) + i*sin(x)
    # E[exp(ix)] = E[cos(x)] + i*E[sin(x)]
    cos_mean = torch.cos(args).mean(dim=0)
    sin_mean = torch.sin(args).mean(dim=0)

    # 4. Loss
    # |ECF - Target|^2 = |(Real - Target) + i(Imag)|^2 = (Real - Target)^2 + Imag^2
    # Target (exp_f) is real-valued.
    diff_sq = (cos_mean - exp_f.unsqueeze(0)).square() + sin_mean.square()

    err = diff_sq * exp_f.unsqueeze(0)

    loss = torch.trapz(err, t, dim=1) * N
    return loss.mean()

import torch
import torch.nn as nn
import torch.nn.functional as F

class RotaryEmbedding2D(nn.Module):
    def __init__(self, dim, max_shape=(32, 32)):
        super().__init__()
        self.dim = dim
        # We split dim into two for x and y frequencies
        self.dim_x = dim // 2
        self.dim_y = dim - self.dim_x

        # Precompute frequencies
        inv_freq_x = 1.0 / (10000 ** (torch.arange(0, self.dim_x, 2).float() / self.dim_x))
        inv_freq_y = 1.0 / (10000 ** (torch.arange(0, self.dim_y, 2).float() / self.dim_y))

        self.register_buffer("inv_freq_x", inv_freq_x)
        self.register_buffer("inv_freq_y", inv_freq_y)

    def forward(self, h, w, device):
        # Generate grid
        seq_y = torch.arange(h, device=device, dtype=self.inv_freq_y.dtype)
        seq_x = torch.arange(w, device=device, dtype=self.inv_freq_x.dtype)

        # Outer product to get (H, W, dim/2)
        freqs_x = torch.einsum("i,j->ij", seq_x, self.inv_freq_x)
        freqs_y = torch.einsum("i,j->ij", seq_y, self.inv_freq_y)

        # Combine to (H, W, dim/2) -> repeat for cos/sin format
        emb_x = torch.cat((freqs_x, freqs_x), dim=-1)
        emb_y = torch.cat((freqs_y, freqs_y), dim=-1)

        # We need to construct the full 2D embeddings
        # Assuming we split the head dim: [x_part, y_part]
        # We broaden to fit the sequence length
        # Result shape: [H*W, 1, Dim] for broadcasting

        # Broadcast x along height, y along width
        # freqs_x: [W, dim_x] -> [H, W, dim_x]
        emb_x = emb_x.unsqueeze(0).repeat(h, 1, 1)
        # freqs_y: [H, dim_y] -> [H, W, dim_y]
        emb_y = emb_y.unsqueeze(1).repeat(1, w, 1)

        # Concatenate x and y frequencies: [H, W, dim]
        freqs = torch.cat([emb_x, emb_y], dim=-1)

        # Flatten: [H*W, dim]
        freqs = freqs.flatten(0, 1)
        return freqs[None, :, :] # [1, Seq, Dim]

def apply_rotary_pos_emb(q, k, freqs):
    # q, k: [B, H, Seq, Dim]
    # freqs: [1, Seq, Dim]

    # Split into pairs for rotation
    q_len = q.shape[-1]

    # Cos/Sin
    cos = freqs.cos()
    sin = freqs.sin()

    # Apply rotation
    # (x, y) -> (x cos - y sin, x sin + y cos)
    # Standard rotate_half implementation
    def rotate_half(x):
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat((-x2, x1), dim=-1)

    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

import torch.nn.functional as F

def drop_path(x, drop_prob: float = 0., training: bool = False):
    if drop_prob == 0. or not training: return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()
    return x.div(keep_prob) * random_tensor

class DropPath(nn.Module):
    def __init__(self, drop_prob=None):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob
    def forward(self, x): return drop_path(x, self.drop_prob, self.training)

class ThermoAttention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        # Note: self.scale is handled automatically by SDPA, but good to keep if needed manually
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(DROPOUT)

        # RoPE generator
        self.rope = RotaryEmbedding2D(head_dim)

        self.q_norm = nn.RMSNorm(head_dim, eps=1e-6)
        self.k_norm = nn.RMSNorm(head_dim, eps=1e-6)

    def forward(self, x, H, W):
        B, N, C = x.shape
        # Shape: [3, B, Heads, SeqLen, HeadDim]
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        q = self.q_norm(q)
        k = self.k_norm(k)

        cls_q = q[:, :, :1]
        cls_k = k[:, :, :1]

        q = q[:, :, 1:]
        k = k[:, :, 1:]

        # --- Apply 2D RoPE ---
        # RoPE modifies Q and K in place or returns new tensors.
        # It operates on the HeadDim, so it's compatible with the split heads.
        freqs = self.rope(H, W, x.device) # [1, SeqLen, HeadDim]
        q, k = apply_rotary_pos_emb(q, k, freqs)
        # ---------------------

        q = torch.cat((cls_q, q), dim=2)
        k = torch.cat((cls_k, k), dim=2)

        # --- Flash Attention ---
        # PyTorch 2.0+ automatically optimizes this using FlashAttention v2 on CUDA.
        # Input shapes are already (Batch, Heads, SeqLen, Dim), which SDPA expects.
        x = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=DROPOUT,
            is_causal=False  # ViT is bidirectional, not causal like GPT
        )

        # Reshape back: [B, Heads, N, Dim] -> [B, N, C]
        x = x.transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        # ---------------------
        return x

class ThermoViTBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4., reg_mode='baseline', sketch_dim=64):
        super().__init__()
        self.norm1 = nn.RMSNorm(dim, eps=1e-6)
        self.attn = ThermoAttention(dim, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dim, eps=1e-6)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(int(dim * mlp_ratio), dim),
            nn.Dropout(DROPOUT),
        )
        self.reg_mode = reg_mode
        self.sketch_dim = sketch_dim

        self.drop_path = DropPath(DROPOUT) if DROPOUT > 0. else nn.Identity()

        self.gamma_1 = nn.Parameter(1e-4 * torch.ones((dim)),requires_grad=True)
        self.gamma_2 = nn.Parameter(1e-4 * torch.ones((dim)),requires_grad=True)

    def forward(self, x, H, W):
        # Attention Residual
        x = x + self.gamma_1 * self.drop_path(self.attn(self.norm1(x), H, W))

        # MLP Residual
        # Note: We apply SIGReg AFTER the block computation but BEFORE the next block.
        # This keeps the "Residual Stream" clean and Gaussian.

        x = x + self.gamma_2 * self.drop_path(self.mlp(self.norm2(x)))

        # --- PHYSICS INJECTION ---
        reg_loss = torch.tensor(0.0, device=x.device)
        if self.reg_mode != 'baseline':
            # Global Average Pool of the tokens [B, N, C] -> [B, C]
            # This represents the "Image Vector" at this depth
            flat_rep = x.mean(dim=1)

            # Crucial: Pre-Norm vs Post-Norm context.
            # LayerNorm forces variance=1. SIGReg forces Distribution=Gaussian.
            # They are compatible.
            if self.reg_mode == 'weak':
                reg_loss = sigreg_weak_loss(flat_rep, self.sketch_dim)
            elif self.reg_mode == 'strong':
                reg_loss = sigreg_strong_loss(flat_rep, self.sketch_dim)

        return x, reg_loss

class ThermoViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, num_classes=100,
                 dim=384, depth=12, heads=12, mlp_ratio=4,
                 reg_mode='strong', sketch_dim=64):
        super().__init__()

        self.img_size = img_size
        self.patch_size = patch_size
        self.H = img_size // patch_size
        self.W = img_size // patch_size
        num_patches = self.H * self.W

        # Patch Embedding
        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)

        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, dim))

        # Blocks
        self.blocks = nn.ModuleList([
            ThermoViTBlock(dim, heads, mlp_ratio, reg_mode, sketch_dim)
            for _ in range(depth)
        ])

        self.norm = nn.RMSNorm(dim, eps=1e-6)
        self.head = nn.Linear(dim, num_classes)

        # Initialize weights (trunc_normal is usually good for ViT)
        self.apply(self._init_weights)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            torch.nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x):
        B = x.shape[0]

        # Patch Embed: [B, C, H, W] -> [B, N, C]
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # Prepend CLS
        x = x + self.pos_embed

        total_phys_loss = 0.0

        # Pass through blocks
        for blk in self.blocks:
            x, l_loss = blk(x, self.H, self.W)
            total_phys_loss += l_loss

        # Classifier
        x = self.norm(x)
        out = self.head(x[:, 0])

        return out, (total_phys_loss / len(self.blocks))

def ViT():
    return ThermoViT(
        img_size=IMG_SIZE,
        patch_size=PATCH_SIZE,
        num_classes=NUM_CLASSES,
        dim=192,
        depth=12,
        heads=3,
        mlp_ratio=4,
        reg_mode=REG_MODE,
        sketch_dim=SKETCH_DIM
    )


# ==========================================
# 4. Data Loading
# ==========================================

class HFImageNetWrapper(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform
    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image']
        if image.mode != 'RGB': image = image.convert('RGB')
        if self.transform: image = self.transform(image)
        return image, item['label']

def get_hf_dataloaders(batch_size):
    print("==> Loading HF Dataset...")
    hf_ds = load_dataset("benjamin-paine/imagenet-1k-256x256", num_proc=4)
    mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    train_transform = transforms.Compose([
        transforms.RandomCrop(224, padding=4, padding_mode='reflect'),
        transforms.RandomHorizontalFlip(),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    val_transform = transforms.Compose([
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    train_set = HFImageNetWrapper(hf_ds['train'], transform=train_transform)
    val_set = HFImageNetWrapper(hf_ds['validation'], transform=val_transform)

    # We use persistent workers to keep things fast
    train_loader = torch.utils.data.DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=4, pin_memory=True, persistent_workers=True
    )
    val_loader = torch.utils.data.DataLoader(
        val_set, batch_size=batch_size, shuffle=False,
        num_workers=4, pin_memory=True, persistent_workers=True
    )
    return train_loader, val_loader

# ==========================================
# 5. Helpers for Step-Based Logic
# ==========================================

def cycle(iterable):
    """Makes a dataloader infinite."""
    while True:
        for x in iterable:
            yield x

@torch.no_grad()
def evaluate(net, testloader, criterion, step, best_acc):
    net.eval()
    test_loss = 0
    correct = 0
    total = 0

    print(f"--> Starting Evaluation at Step {step}...")

    # Iterate through the whole validation set
    for batch_idx, (inputs, targets) in enumerate(testloader):
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        outputs, _ = net(inputs)
        loss = criterion(outputs, targets)

        test_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    acc = 100. * correct / total
    avg_loss = test_loss / (batch_idx + 1)

    if acc > best_acc:
        best_acc = acc
        torch.save(net.state_dict(), f'vit_tiny_{REG_MODE}_best.pth')

    print(f"--> Eval Result | Step: {step} | Loss: {avg_loss:.4f} | Acc: {acc:.2f}% | Best: {best_acc:.2f}%")

    wandb.log({
        "step": step,
        "val_loss": avg_loss,
        "val_acc": acc,
        "best_acc": best_acc
    })

    net.train()
    return best_acc

# ==========================================
# 6. Main Step-Based Training Loop
# ==========================================
if __name__ == '__main__':
    trainloader, testloader = get_hf_dataloaders(BATCH_SIZE)

    # Calculate Max Steps
    steps_per_epoch = len(trainloader)
    MAX_STEPS = EPOCHS * steps_per_epoch
    print(f"==> Total Training Steps: {MAX_STEPS} (approx {EPOCHS} epochs)")

    print(f'==> Building ViT-Tiny/16...')
    net = ViT().to(DEVICE)
    net = torch.compile(net)

    hidden_weights = [p for p in net.parameters() if p.ndim == 2]
    non_hidden_params = [p for p in net.parameters() if p.ndim != 2]

    # Note: LinearLR and CosineAnnealingLR scale the LR defined here relative to 1.0
    param_groups = [
      dict(params=hidden_weights, use_muon=True, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY),
      dict(params=non_hidden_params, use_muon=False, lr=LEARNING_RATE, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY)
    ]

    criterion = nn.CrossEntropyLoss()
    optimizer = SingleDeviceMuonWithAuxAdam(param_groups)

    def get_lr(step):
        if step < warmup_steps:
            return step / warmup_steps
        else:
            return 0.5 * (1 + np.cos(np.pi * (step - warmup_steps) / (MAX_STEPS - warmup_steps)))

    warmup_steps = 5 * steps_per_epoch  # 5 epochs warmup
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr)

    # ---------------------------

    train_iterator = cycle(trainloader)

    net.train()
    best_acc = 0.0

    m_loss, m_acc, m_phys, m_samples = 0.0, 0.0, 0.0, 0
    start_time = time.time()

    print("==> Starting Training...")

    for global_step in range(1, MAX_STEPS + 1):

        # 1. Get Batch
        inputs, targets = next(train_iterator)
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

        # Apply Mixup/CutMix
        r = np.random.rand(1)
        if r < 0.5: # Mixup
            lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
            index = torch.randperm(inputs.size(0)).to(DEVICE)
            inputs = lam * inputs + (1 - lam) * inputs[index, :]
            targets_a, targets_b = targets, targets[index]
        else: # CutMix
            lam = np.random.beta(CUTMIX_ALPHA, CUTMIX_ALPHA)
            rand_index = torch.randperm(inputs.size(0)).to(DEVICE)
            target_a = targets
            target_b = targets[rand_index]
            bbx1, bby1, bbx2, bby2 = rand_bbox(inputs.size(), lam)
            inputs[:, :, bbx1:bbx2, bby1:bby2] = inputs[rand_index, :, bbx1:bbx2, bby1:bby2]
            lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (inputs.size()[-1] * inputs.size()[-2]))
            targets_a, targets_b = target_a, target_b

        optimizer.zero_grad()

        # 3. Forward & Backward
        outputs, p_loss = net(inputs)
        c_loss = criterion(outputs, targets_a) * lam + criterion(outputs, targets_b) * (1. - lam)

        sig_weight = min(1.0, global_step / (5 * steps_per_epoch)) * SIGR_ALPHA
        loss = (1 - sig_weight) * c_loss + (sig_weight * p_loss)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        optimizer.step()

        # 4. Schedule per step
        scheduler.step()

        # 5. Metrics Accumulation
        _, predicted = outputs.max(1)
        acc_batch = (lam * predicted.eq(targets_a).float() + (1 - lam) * predicted.eq(targets_b).float()).sum().item()

        m_loss += c_loss.item()
        m_phys += p_loss.item()
        m_acc += acc_batch
        m_samples += targets.size(0)

        # 6. Logging
        if global_step % LOG_INTERVAL == 0:
            elapsed = time.time() - start_time

            # Average over the interval
            log_loss = m_loss / LOG_INTERVAL
            log_acc = 100.0 * m_acc / m_samples
            log_phys = m_phys / LOG_INTERVAL
            current_lr = scheduler.get_last_lr()[0]

            wandb.log({
                "step": global_step,
                "train_loss": log_loss,
                "train_acc": log_acc,
                "phys_loss": log_phys,
                "lr": current_lr,
                "time": elapsed
            })

            print(f"Step {global_step}/{MAX_STEPS} | Loss: {log_loss:.4f} | Acc: {log_acc:.1f}% | Phys: {log_phys:.2f} | Time: {elapsed:.0f}s")

            # Reset meters
            m_loss, m_acc, m_phys, m_samples = 0.0, 0.0, 0.0, 0

            start_time = time.time()

        # 7. Evaluation
        if global_step % EVAL_INTERVAL == 0 or global_step == MAX_STEPS:
            best_acc = evaluate(net, testloader, criterion, global_step, best_acc)
            # Reset timer to avoid counting eval time in training speed metric (optional)
            start_time = time.time()

    print(f"Final Best Accuracy: {best_acc:.2f}%")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: akbar2habibullah (akbar2habibullah-kreasof-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Training on: cuda | Mode: strong | Alpha: 0.001
==> Loading HF Dataset...


Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/38 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
Step 756100/1001000 | Loss: 3.3201 | Acc: 39.3% | Phys: 49.48 | Time: 28s
Step 756150/1001000 | Loss: 3.1884 | Acc: 41.2% | Phys: 50.89 | Time: 28s
Step 756200/1001000 | Loss: 3.3998 | Acc: 38.9% | Phys: 51.53 | Time: 28s
Step 756250/1001000 | Loss: 3.3365 | Acc: 38.9% | Phys: 49.44 | Time: 28s
Step 756300/1001000 | Loss: 3.3872 | Acc: 38.2% | Phys: 49.55 | Time: 28s
Step 756350/1001000 | Loss: 3.4115 | Acc: 37.8% | Phys: 49.16 | Time: 28s
Step 756400/1001000 | Loss: 3.3645 | Acc: 38.7% | Phys: 50.06 | Time: 28s
Step 756450/1001000 | Loss: 3.4723 | Acc: 36.7% | Phys: 51.40 | Time: 28s
Step 756500/1001000 | Loss: 3.4465 | Acc: 37.2% | Phys: 51.35 | Time: 28s
Step 756550/1001000 | Loss: 3.2813 | Acc: 39.9% | Phys: 51.04 | Time: 28s
Step 756600/1001000 | Loss: 3.3950 | Acc: 38.0% | Phys: 51.35 | Time: 28s
Step 756650/1001000 | Loss: 3.3419 | Acc: 38.5% | Phys: 52.44 | Time: 28s
Step 756700/1001000 | Loss: 3.0143 | Acc: 43.0% | Phys: 51.53